# Predictive Maintenance – ML Training (Telemetry)

This notebook prepares the **Predictive ML layer** for the hybrid maintenance system: we explore the synthetic telemetry dataset, select features that align with operational/telemetry inputs, preprocess the data, train several models (Random Forest, XGBoost, Logistic Regression, Gradient Boosting), compare them, and save the best model(s) for use by the decision engine.

**Dependencies:** `pandas`, `scikit-learn`, `joblib`; `xgboost` is optional (install with `pip install xgboost` to train XGBoost). All are listed in the project `requirements.txt`.

## 1. Load the dataset and inspect structure

Load the CSV and check shape, column types, and first rows so we know what we are working with.

In [1]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("synthetic_telemetry_data.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nDtypes:")
print(df.dtypes)
df.head()

Shape: (1970, 41)

Columns: ['vehicle_id', 'brand', 'timestamp', 'odometer_reading', 'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c', 'fuel_level_percent', 'fuel_consumption_lph', 'engine_load_percent', 'throttle_pos_percent', 'air_flow_rate_gps', 'exhaust_gas_temp_c', 'vibration_level', 'engine_hours', 'brake_fluid_level_psi', 'brake_pad_wear_mm', 'brake_temp_c', 'abs_fault_indicator', 'brake_pedal_pos_percent', 'wheel_speed_fl_kph', 'wheel_speed_fr_kph', 'wheel_speed_rl_kph', 'wheel_speed_rr_kph', 'battery_voltage_v', 'battery_current_a', 'battery_temp_c', 'alternator_output_v', 'battery_charge_percent', 'battery_health_percent', 'vehicle_speed_kph', 'ambient_temp_c', 'humidity_percent', 'gps_latitude', 'gps_longitude', 'engine_failure_imminent', 'brake_issue_imminent', 'battery_issue_imminent', 'failure_date', 'failure_type']

Dtypes:
vehicle_id                  object
brand                       object
timestamp                   object
odometer_reading         

,vehicle_id,brand,timestamp,odometer_reading,engine_temp_c,engine_rpm,oil_pressure_psi,coolant_temp_c,fuel_level_percent,fuel_consumption_lph,...,vehicle_speed_kph,ambient_temp_c,humidity_percent,gps_latitude,gps_longitude,engine_failure_imminent,brake_issue_imminent,battery_issue_imminent,failure_date,failure_type
0,VEH0000,BMW,2023-01-01 00:00:00,31381.561743,99.388213,2212.155243,55.522505,91.788105,68.368162,4.076315,...,116.501239,24.187057,83.785796,31.986296,78.203395,0,0,0,2050-01-01 00:00:00,No Failure
1,VEH0000,BMW,2023-01-01 00:56:00,31391.284155,84.118706,961.795251,57.799346,87.283282,99.064394,7.059168,...,39.409933,15.148875,57.418501,30.620539,84.389727,0,0,0,2050-01-01 00:00:00,No Failure
2,VEH0000,BMW,2023-01-01 01:28:00,31428.088015,103.521431,742.050437,50.398848,83.503155,80.280244,10.792362,...,82.077798,23.138714,80.540135,30.634997,79.039234,0,0,0,2050-01-01 00:00:00,No Failure
3,VEH0000,BMW,2023-01-01 01:27:00,31463.271669,91.130527,2723.042461,41.719715,90.997963,13.892586,1.564630,...,70.294428,35.058947,57.221563,33.490971,76.107248,0,0,0,2050-01-01 00:00:00,No Failure
4,VEH0000,BMW,2023-01-01 01:00:00,31490.411858,96.887263,2647.979969,38.438964,93.029011,20.871931,6.814662,...,74.028392,18.024207,47.680115,28.392300,83.160007,0,0,0,2050-01-01 00:00:00,No Failure


## 2. Explore target variables and class balance

The dataset has three binary flags (`engine_failure_imminent`, `brake_issue_imminent`, `battery_issue_imminent`) and a `failure_type` label. We need to understand imbalance so we can choose metrics and handling (e.g. stratified split, class weights).

In [2]:
print("Target value counts:")
for col in ["engine_failure_imminent", "brake_issue_imminent", "battery_issue_imminent"]:
    print(f"  {col}:", df[col].value_counts().to_dict())
print("\nfailure_type:")
print(df["failure_type"].value_counts())

# Combined target: any component failure imminent (for overall risk)
df["any_failure_imminent"] = (
    df["engine_failure_imminent"] | df["brake_issue_imminent"] | df["battery_issue_imminent"]
).astype(int)
print("\nany_failure_imminent:", df["any_failure_imminent"].value_counts().to_dict())

Target value counts:
  engine_failure_imminent: {0: 1963, 1: 7}
  brake_issue_imminent: {0: 1954, 1: 16}
  battery_issue_imminent: {0: 1958, 1: 12}

failure_type:
failure_type
No Failure             1935
Brake Pad Worn            9
Low Brake Fluid           5
Battery Dead              4
Engine Overheat           4
Battery Drain             4
Low Battery Voltage       4
Excessive Vibration       2
Brake Overheat            2
Low Oil Pressure          1
Name: count, dtype: int64

any_failure_imminent: {0: 1935, 1: 35}


## 3. Choose features relevant to telemetry-based risk

We keep features that match the system's operational inputs and telemetry (engine RPM, engine temp, oil pressure, battery voltage, fuel consumption, mileage, etc.) and drop identifiers, GPS, and raw target columns. This keeps the ML layer focused on telemetry rather than duplicating rule logic.

In [3]:
ID_COLS = ["vehicle_id", "brand", "timestamp", "failure_date", "failure_type"]
TARGET_COLS = ["engine_failure_imminent", "brake_issue_imminent", "battery_issue_imminent"]
DROP_COLS = ID_COLS + TARGET_COLS + ["gps_latitude", "gps_longitude"]  # GPS not used for risk in demo

feature_cols = [c for c in df.columns if c not in DROP_COLS and c != "any_failure_imminent"]
print("Selected feature columns (", len(feature_cols), "):", feature_cols)
X = df[feature_cols].copy()
y_overall = df["any_failure_imminent"]
y_engine = df["engine_failure_imminent"]
y_brake = df["brake_issue_imminent"]
y_battery = df["battery_issue_imminent"]

Selected feature columns ( 31 ): ['odometer_reading', 'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c', 'fuel_level_percent', 'fuel_consumption_lph', 'engine_load_percent', 'throttle_pos_percent', 'air_flow_rate_gps', 'exhaust_gas_temp_c', 'vibration_level', 'engine_hours', 'brake_fluid_level_psi', 'brake_pad_wear_mm', 'brake_temp_c', 'abs_fault_indicator', 'brake_pedal_pos_percent', 'wheel_speed_fl_kph', 'wheel_speed_fr_kph', 'wheel_speed_rl_kph', 'wheel_speed_rr_kph', 'battery_voltage_v', 'battery_current_a', 'battery_temp_c', 'alternator_output_v', 'battery_charge_percent', 'battery_health_percent', 'vehicle_speed_kph', 'ambient_temp_c', 'humidity_percent']


## 4. Check for missing values and basic stats

Ensure we have no unexpected nulls and review distributions so we can decide on imputation or dropping if needed.

In [4]:
print("Missing per column:")
print(X.isnull().sum())
print("\nDescribe (numeric):")
X.describe()

Missing per column:
odometer_reading           0
engine_temp_c              0
engine_rpm                 0
oil_pressure_psi           0
coolant_temp_c             0
fuel_level_percent         0
fuel_consumption_lph       0
engine_load_percent        0
throttle_pos_percent       0
air_flow_rate_gps          0
exhaust_gas_temp_c         0
vibration_level            0
engine_hours               0
brake_fluid_level_psi      0
brake_pad_wear_mm          0
brake_temp_c               0
abs_fault_indicator        0
brake_pedal_pos_percent    0
wheel_speed_fl_kph         0
wheel_speed_fr_kph         0
wheel_speed_rl_kph         0
wheel_speed_rr_kph         0
battery_voltage_v          0
battery_current_a          0
battery_temp_c             0
alternator_output_v        0
battery_charge_percent     0
battery_health_percent     0
vehicle_speed_kph          0
ambient_temp_c             0
humidity_percent           0
dtype: int64

Describe (numeric):


,odometer_reading,engine_temp_c,engine_rpm,oil_pressure_psi,coolant_temp_c,fuel_level_percent,fuel_consumption_lph,engine_load_percent,throttle_pos_percent,air_flow_rate_gps,...,wheel_speed_rr_kph,battery_voltage_v,battery_current_a,battery_temp_c,alternator_output_v,battery_charge_percent,battery_health_percent,vehicle_speed_kph,ambient_temp_c,humidity_percent
count,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,...,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000,1970.000000
mean,55805.596064,95.098185,2011.461245,44.705878,89.874554,47.998493,5.123180,39.999500,29.981324,50.292807,...,60.083571,12.578856,10.090152,25.003929,14.197447,70.092167,94.329009,70.201288,29.966583,59.783230
std,25276.174846,5.201481,760.883948,10.115871,2.985144,29.182622,2.919474,19.273029,18.799032,19.670713,...,30.272161,0.531746,21.207427,5.098852,0.292889,17.161083,2.541468,38.185877,9.867784,14.721301
min,13578.948264,77.781925,700.000000,5.928049,78.797191,0.166009,0.000000,0.000000,0.000000,0.000000,...,-42.231741,9.065082,-67.829694,6.669371,13.110096,40.018290,90.162163,0.000000,-17.116179,9.914752
25%,34537.434560,91.593183,1457.167122,37.687975,87.853215,22.081471,2.990431,26.540634,16.268606,36.972980,...,39.548633,12.266905,-3.473606,21.494980,14.007229,55.096188,91.960024,42.645593,23.609246,50.103555
50%,56545.906171,95.084360,1998.276722,45.024308,89.767156,47.287384,5.052483,39.383741,28.781407,50.179637,...,60.502644,12.595619,10.737516,25.002662,14.206000,70.652447,94.172950,69.336619,30.271354,59.574084
75%,77938.260730,98.620236,2535.801415,51.481993,91.926039,73.024963,7.132336,53.384306,42.784640,64.408787,...,80.145359,12.928926,24.296475,28.411607,14.394783,84.273524,96.249211,96.188556,36.709522,69.550834
max,100409.771733,119.322471,4167.823670,80.267052,100.659740,99.997479,15.127696,100.000000,91.815640,106.856022,...,160.179239,14.524733,83.362796,42.367341,15.241447,99.968430,99.386264,180.000000,63.348462,117.766913


## 5. Preprocess: handle missing values and prepare feature matrix

Fill any missing numeric values with the column median so tree models get a consistent matrix. Then split into train/test with stratification on the overall target to preserve class balance.

In [5]:
from sklearn.model_selection import train_test_split

X_filled = X.fillna(X.median())
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_overall, test_size=0.2, random_state=42, stratify=y_overall
)
print("Train size:", X_train.shape[0], "Test size:", X_test.shape[0])
print("Train positive rate:", y_train.mean())
print("Test positive rate:", y_test.mean())

Train size: 1576 Test size: 394
Train positive rate: 0.017766497461928935
Test positive rate: 0.017766497461928935


## 6. Train multiple models – baseline and tree ensembles

Train Logistic Regression (baseline), Random Forest, XGBoost, and Gradient Boosting. Use class_weight='balanced' (or equivalent) to handle imbalance. We compare on ROC-AUC and F1 to account for the rare positive class.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

def train_eval(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1] if hasattr(model, "predict_proba") else pred
    auc = roc_auc_score(y_te, proba) if y_te.nunique() > 1 else 0.0
    f1 = f1_score(y_te, pred, zero_division=0)
    print(f"--- {name} ---")
    print(classification_report(y_te, pred, zero_division=0))
    print(f"ROC-AUC: {auc:.4f}  F1: {f1:.4f}")
    return {"name": name, "model": model, "auc": auc, "f1": f1}

results = []

results.append(train_eval(
    "Logistic Regression",
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    X_train, y_train, X_test, y_test
))
results.append(train_eval(
    "Random Forest",
    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    X_train, y_train, X_test, y_test
))
results.append(train_eval(
    "Gradient Boosting",
    GradientBoostingClassifier(n_estimators=100, random_state=42),
    X_train, y_train, X_test, y_test
))
if HAS_XGB:
    results.append(train_eval(
        "XGBoost",
        xgb.XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric="logloss", scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1), random_state=42),
        X_train, y_train, X_test, y_test
    ))

c:\Users\bolim\miniconda3\envs\malwarepred\lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


--- Logistic Regression ---
              precision    recall  f1-score   support

           0       1.00      0.93      0.96       387
           1       0.18      0.86      0.30         7

    accuracy                           0.93       394
   macro avg       0.59      0.89      0.63       394
weighted avg       0.98      0.93      0.95       394

ROC-AUC: 0.9324  F1: 0.3000
--- Random Forest ---
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       387
           1       1.00      0.29      0.44         7

    accuracy                           0.99       394
   macro avg       0.99      0.64      0.72       394
weighted avg       0.99      0.99      0.98       394

ROC-AUC: 0.9016  F1: 0.4444
--- Gradient Boosting ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       387
           1       1.00      0.86      0.92         7

    accuracy                           1.00       

## 7. Compare models and select the best

Rank by ROC-AUC (and F1) to choose which model to save for the decision engine. We prefer a model that gives a probability for failure risk.

In [7]:
comparison = pd.DataFrame([{"name": r["name"], "ROC-AUC": r["auc"], "F1": r["f1"]} for r in results]).set_index("name")
comparison = comparison.sort_values("ROC-AUC", ascending=False)
print(comparison)
best = max(results, key=lambda r: r["auc"])
print(f"\nBest model by ROC-AUC: {best['name']}")

                      ROC-AUC        F1
name                                   
XGBoost              0.999631  0.857143
Logistic Regression  0.932447  0.300000
Gradient Boosting    0.928571  0.923077
Random Forest        0.901624  0.444444

Best model by ROC-AUC: XGBoost


## 8. Save the best model and feature metadata

Persist the best model (e.g. joblib) and the list of feature names so the prediction service can load the same model and align inputs to the same columns.

In [8]:
import joblib

OUT_DIR = Path("data")
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = OUT_DIR / "model.pkl"
FEATURES_PATH = OUT_DIR / "model_features.txt"

joblib.dump(best["model"], MODEL_PATH)
with open(FEATURES_PATH, "w") as f:
    f.write("\n".join(feature_cols))
print("Saved model to", MODEL_PATH)
print("Saved feature list to", FEATURES_PATH)
print("Feature order:", feature_cols)

Saved model to data\model.pkl
Saved feature list to data\model_features.txt
Feature order: ['odometer_reading', 'engine_temp_c', 'engine_rpm', 'oil_pressure_psi', 'coolant_temp_c', 'fuel_level_percent', 'fuel_consumption_lph', 'engine_load_percent', 'throttle_pos_percent', 'air_flow_rate_gps', 'exhaust_gas_temp_c', 'vibration_level', 'engine_hours', 'brake_fluid_level_psi', 'brake_pad_wear_mm', 'brake_temp_c', 'abs_fault_indicator', 'brake_pedal_pos_percent', 'wheel_speed_fl_kph', 'wheel_speed_fr_kph', 'wheel_speed_rl_kph', 'wheel_speed_rr_kph', 'battery_voltage_v', 'battery_current_a', 'battery_temp_c', 'alternator_output_v', 'battery_charge_percent', 'battery_health_percent', 'vehicle_speed_kph', 'ambient_temp_c', 'humidity_percent']


## 9. (Optional) Component-level risk models

For component-level risk (engine, brake, battery), we can train small binary models per component and save them so the prediction service can return per-component risks alongside overall risk.

In [9]:
from sklearn.ensemble import RandomForestClassifier

component_models = {}
for label, y in [("engine", y_engine), ("brake", y_brake), ("battery", y_battery)]:
    Xt, Xv, yt, yv = train_test_split(X_filled, y, test_size=0.2, random_state=42, stratify=y)
    clf = RandomForestClassifier(n_estimators=50, class_weight="balanced", random_state=42)
    clf.fit(Xt, yt)
    component_models[label] = clf
    print(f"{label}: train pos rate {yt.mean():.3f}, test AUC {roc_auc_score(yv, clf.predict_proba(Xv)[:, 1]):.3f}")

joblib.dump(component_models, OUT_DIR / "model_components.pkl")
print("Saved component models to", OUT_DIR / "model_components.pkl")

engine: train pos rate 0.004, test AUC 1.000
brake: train pos rate 0.008, test AUC 1.000
battery: train pos rate 0.006, test AUC 0.737
Saved component models to data\model_components.pkl
